In [ ]:
#!pip install sentence_transformers
#!pip install ag2[retrievechat]
#!pip install pyautogen==0.7.2
#https://microsoft.github.io/autogen/0.2/docs/notebooks/agentchat_RetrieveChat/

In [ ]:
import json
import os

import chromadb

import autogen
from autogen import AssistantAgent
from autogen.agentchat.contrib.retrieve_user_proxy_agent import RetrieveUserProxyAgent

# Accepted file formats for that can be stored in
# a vector database instance
from autogen.retrieve_utils import TEXT_FORMATS

config_list = autogen.config_list_from_json("OAI_CONFIG_LIST.json")

assert len(config_list) > 0
print("models to use: ", [config_list[i]["model"] for i in range(len(config_list))])

In [ ]:
print("Accepted file formats for `docs_path`:")
print(TEXT_FORMATS)

In [ ]:
config_list = [
    {
        "model": "gpt-4",
        "api_key": "putyourAPikey",
        "api_type": "openai"
    }
]

In [ ]:
import agentops
agentops.init("putyourkey")

In [ ]:
# 1. create an AssistantAgent instance named "assistant"
assistant = AssistantAgent(
    name="assistant",
    system_message="You are a helpful assistant.",
    llm_config={
        "timeout": 600,
        "cache_seed": 42,
        "config_list": config_list,
    },
)


In [ ]:
from autogen.agentchat.contrib.retrieve_user_proxy_agent import RetrieveUserProxyAgent

# Extract text from local PDF files
pdf_files = ["book.pdf"]
#docs_content = [extract_text_from_pdf(file) for file in pdf_files]

from autogen.retrieve_utils import TEXT_FORMATS,extract_text_from_pdf
docs_content= extract_text_from_pdf("book.pdf")

In [ ]:
# Define the RetrieveUserProxyAgent instance
ragproxyagent = RetrieveUserProxyAgent(
    name="ragproxyagent",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=3,
    retrieve_config={
        "task": "qa",
        "docs_path": ["autogen-docs/book.pdf"],
        "docs_content": docs_content,  # Provide the extracted content directly
        "chunk_token_size": 200,
        "model": config_list[0]["model"],
        "vector_db": "chroma",
        "overwrite": True,
        "get_or_create": True,
    },
    code_execution_config=False,

In [ ]:
# Process Starts for Scenario 1 : Generate code based off docstrings w/o human feedback

In [ ]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

qa_problem = "summarize the content and use python code to any calculation given in the book"
chat_result = ragproxyagent.initiate_chat(assistant, message=ragproxyagent.message_generator, problem=qa_problem)
agentops.end_session("Success")

In [ ]:
#agentops.end_session("Success")

In [ ]:
# Process Starts for Scenario 2 : Answer a question based off docstrings w/o human feedback

In [ ]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

qa_problem = "You are a creative head . do many brainstorms"
chat_result = ragproxyagent.initiate_chat(assistant, message=ragproxyagent.message_generator, problem=qa_problem)

In [ ]:
# Process Starts for Scenario 3 : Generate code based off docstrings w/ human feedback

In [ ]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

# set `human_input_mode` to be `ALWAYS`, so the agent will ask for human input at every step.
ragproxyagent.human_input_mode = "ALWAYS"
code_problem = "Based on this book give Marketing idea for AI and Datascience trainig institute. Give 5 ideas. if any coding requires use python and save it as .py"
chat_result = ragproxyagent.initiate_chat(assistant, message=ragproxyagent.message_generator, problem=code_problem)

In [ ]:
# Process Starts for Scenario 4: Answer a question based off docstrings w/ human feedback

In [ ]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

# set `human_input_mode` to be `ALWAYS`, so the agent will ask for human input at every step.
ragproxyagent.human_input_mode = "ALWAYS"
qa_problem = "Based on this book give Marketing idea for AI and Datascience trainig institute. Give 5 ideas. "
chat_result = ragproxyagent.initiate_chat(
    assistant, message=ragproxyagent.message_generator, problem=qa_problem
)  # type "exit" to exit the conversation